# Shortest paths in navigable graphs

Dijkstra's algorithm over a directed graph given as an adjacency list of the form

```
<source_id> [(neighbor, uncov), (neighbor, uncov), ...]
```

one source per line (the same format produced by `edge_to_coverage_analysis.py` /
`distributed_robust_prune.py`). The `uncov` field is ignored here.

**Edge weights.** The weight of an edge `u -> v` is the out-degree of the
destination vertex: `w(u -> v) = deg(v)`, where `deg(v)` is the number of
out-neighbours of `v`. This depends only on the in-vertex `v`, not on `u`, so
every edge entering `v` costs the same.

In [1]:
import ast
import heapq
from tqdm import tqdm

In [2]:
def load_adjacency(path):
    """Load a directed graph from an adjacency-list file.

    Each line is either
        '<source_id> [(neighbor, uncov), ...]'   (source-prefixed), or
        '[(neighbor, uncov), ...]'               (bare; line index is the source).
    The `uncov` field is ignored. Returns adj: dict source -> list of neighbours.
    """
    adj = {}
    with open(path, 'r') as f:
        for i, line in enumerate(tqdm(f, desc='Loading graph')):
            line = line.strip()
            if not line:
                continue
            if line.startswith('['):
                source = i
                neighborhood = ast.literal_eval(line)
            else:
                space = line.index(' ')
                source = int(line[:space])
                neighborhood = ast.literal_eval(line[space + 1:])
            # neighborhood is [(neighbor, uncov), ...]; keep only the neighbour.
            adj[source] = [int(nbr) for (nbr, _uncov) in neighborhood]
    return adj


def out_degrees(adj):
    """deg(v) = number of out-neighbours of v. Vertices that only ever appear as a
    neighbour (never as a source line) have out-degree 0."""
    deg = {u: len(nbrs) for u, nbrs in adj.items()}
    for nbrs in adj.values():
        for v in nbrs:
            deg.setdefault(v, 0)
    return deg

In [ ]:
def dijkstra(adj, deg, source, target=None):
    """Shortest paths from `source` using edge weight w(u -> v) = deg(v).

    Args:
        adj:    dict u -> list of out-neighbours v.
        deg:    dict v -> out-degree of v (the edge weight for any edge into v).
        source: start vertex.
        target: if given, stop early once it is finalised and return
                (distance, path); otherwise return (dist, prev) for all vertices.

    All weights deg(v) >= 0, so Dijkstra is valid. The source has distance 0.
    """
    dist = {source: 0}
    prev = {source: None}
    visited = set()
    pq = [(0, source)]                      # (distance, vertex) min-heap

    while pq:
        d, u = heapq.heappop(pq)
        if u in visited:
            continue                        # stale heap entry
        visited.add(u)

        if target is not None and u == target:
            return d, _reconstruct(prev, target)

        for v in adj.get(u, ()):
            if v in visited:
                continue
            nd = d + deg[v]                 # weight of edge u -> v is deg(v)
            if v not in dist or nd < dist[v]:
                dist[v] = nd
                prev[v] = u
                heapq.heappush(pq, (nd, v))

    if target is not None:
        return float('inf'), None            # target unreachable
    return dist, prev


def _reconstruct(prev, target):
    """Walk `prev` pointers back from target to source, return the forward path."""
    path = []
    node = target
    while node is not None:
        path.append(node)
        node = prev.get(node)
    path.reverse()
    return path


def dijkstra_route(adj, deg, source, target):
    """Shortest path between two vertices, returning the same tuple as greedy_route.

    Returns (reached, weighted_dist, hops, path):
      reached       - True iff target is reachable from source.
      weighted_dist - sum of deg(v) along the optimal path (inf if unreachable).
      hops          - number of edges in the path (0 if unreachable).
      path          - list of vertices [source, ..., target] (None if unreachable).
    """
    d, path = dijkstra(adj, deg, source, target=target)
    if path is None:
        return False, float('inf'), 0, None
    return True, d, len(path) - 1, path


def shortest_path_length(adj, deg, source, target):
    """Number of hops on the shortest (weighted) path; 0 if unreachable."""
    return dijkstra_route(adj, deg, source, target)[2]

## Greedy routing with Euclidean distances

Greedy graph routing toward a target point. Each vertex is a point whose vector
lives in the `'train'` set of an HDF5 file (vertex id = row index), as in
`beam_search/`. From the current node we move to the out-neighbour whose vector
is closest (squared Euclidean) to the target's vector, as long as it strictly
improves on the current node's distance to the target. The walk stops at the
target or at a local minimum (no neighbour closer than the current node).

Unlike Dijkstra, this is a heuristic that follows the graph geometry; it is not
guaranteed to find the target or the shortest path, but it mirrors how a
navigable graph is actually searched.

In [4]:
import numpy as np
import h5py
from scipy.spatial.distance import cdist


def load_vectors(hdf5_path, group='train'):
    """Load the point vectors from an HDF5 file (as in beam_search/).

    Vertex id is the row index into this array. Returns an (n, d) float array.
    """
    with h5py.File(hdf5_path, 'r') as f:
        return f[group][:]

In [ ]:
def greedy_route(adj, deg, X, source, target, max_steps=None):
    """Greedy graph routing from `source` toward `target` using Euclidean distance.

    Each step moves to the out-neighbour whose vector X[v] is closest (squared
    Euclidean) to the target's vector X[target], provided it strictly improves on
    the current node's distance to the target. Stops when it reaches the target,
    or at a local minimum (no neighbour is closer than the current node).

    Args:
        adj:       dict u -> list of out-neighbours v.
        deg:       dict v -> out-degree of v (the edge weight for any edge into v).
        X:         (n, d) array of point vectors; X[i] is vertex i's vector.
        source:    start vertex id (row index into X).
        target:    goal vertex id (row index into X).
        max_steps: optional cap on hops (default: n, a safe upper bound).

    Returns:
        (reached, weighted_dist, hops, path):
          reached       - True iff the walk arrived at target.
          weighted_dist - sum of deg(v) over the edges taken (same weight as
                          Dijkstra), so it's directly comparable.
          hops          - number of edges in the path (= len(path) - 1).
          path          - list of visited vertices [source, ..., last].
    """
    if max_steps is None:
        max_steps = X.shape[0]

    tgt_vec = X[target][None, :]                       # (1, d)
    current = source
    path = [current]
    weighted_dist = 0

    # squared euclidean distance from current node's vector to the target
    cur_dist = float(cdist(X[current][None, :], tgt_vec, metric='sqeuclidean')[0, 0])

    for _ in range(max_steps):
        if current == target:
            break

        nbrs = adj.get(current, [])
        if not nbrs:
            break                                      # dead end

        # distances from each neighbour to the target
        nbr_dists = cdist(X[nbrs], tgt_vec, metric='sqeuclidean').ravel()
        best = int(np.argmin(nbr_dists))
        best_dist = float(nbr_dists[best])

        if best_dist >= cur_dist:
            break                                      # local minimum: no progress

        v = nbrs[best]
        weighted_dist += deg[v]                        # edge current -> v costs deg(v)
        current = v
        cur_dist = best_dist
        path.append(current)

    reached = current == target
    return reached, weighted_dist, len(path) - 1, path


def greedy_path_length(adj, X, source, target, max_steps=None):
    """Back-compat wrapper: (reached, hops) only. Prefer greedy_route."""
    reached, _wd, hops, _path = greedy_route(adj, {}, X, source, target, max_steps)
    return reached, hops

In [6]:
# --- Greedy routing on a real graph + HDF5 vectors ---
# --- Run on a real adjacency-list file ---
ADJ_PATH = "../new_results/adj-list-mnist-euclidean.txt"
HDF5_PATH = '/scratch/pa2439/ANN-Search/datasets/mnist-784-euclidean.hdf5'                           # vectors ('train' set)
adj = load_adjacency(ADJ_PATH)
deg = out_degrees(adj)
X   = load_vectors(HDF5_PATH)          # X[i] is vertex i's vector

Loading graph: 60000it [00:04, 14735.51it/s]


In [ ]:
import pandas as pd

# --- Batch: N random (source, target) pairs; compare Dijkstra vs greedy on
#     both weighted distance (sum of deg(v)) and path length (hops). ---
N_PAIRS = 100
SEED = 0

rng = np.random.default_rng(SEED)
n = X.shape[0]
sources = rng.integers(0, n, size=N_PAIRS)
targets = rng.integers(0, n, size=N_PAIRS)

rows = []
for s, t in tqdm(list(zip(sources, targets)), desc="pairs"):
    s, t = int(s), int(t)
    dj_reached, dj_dist, dj_hops, _ = dijkstra_route(adj, deg, s, t)
    gr_reached, gr_dist, gr_hops, _ = greedy_route(adj, deg, X, s, t)
    rows.append({
        'source': s, 'target': t,
        'dijkstra_reached': dj_reached,
        'dijkstra_weighted_dist': dj_dist,
        'dijkstra_hops': dj_hops,
        'greedy_reached': gr_reached,
        'greedy_weighted_dist': gr_dist,
        'greedy_hops': gr_hops,
    })

results = pd.DataFrame(rows)
results.head()

In [ ]:
# --- Compare on pairs where BOTH methods reached the target (fair comparison) ---
both = results[results['dijkstra_reached'] & results['greedy_reached']].copy()
print(f"{len(both)} / {len(results)} pairs reached by both "
      f"(greedy reached {results['greedy_reached'].mean()*100:.0f}% of all pairs)")

for metric, dj_col, gr_col in [
    ('weighted distance', 'dijkstra_weighted_dist', 'greedy_weighted_dist'),
    ('path length (hops)', 'dijkstra_hops', 'greedy_hops'),
]:
    dj, gr = both[dj_col], both[gr_col]
    ratio = (gr / dj.replace(0, np.nan)).mean()
    print(f"\n{metric}:")
    print(f"  Dijkstra  mean={dj.mean():.2f}  median={dj.median():.1f}")
    print(f"  Greedy    mean={gr.mean():.2f}  median={gr.median():.1f}")
    print(f"  greedy/dijkstra mean ratio = {ratio:.2f}  "
          f"(greedy >= dijkstra on {(gr >= dj).mean()*100:.0f}% of pairs)")

# Scatter: greedy vs dijkstra for each metric, with y=x reference (optimal).
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (title, dj_col, gr_col) in zip(axes, [
    ('Weighted distance', 'dijkstra_weighted_dist', 'greedy_weighted_dist'),
    ('Path length (hops)', 'dijkstra_hops', 'greedy_hops'),
]):
    dj, gr = both[dj_col], both[gr_col]
    ax.scatter(dj, gr, alpha=0.5, s=20)
    lo, hi = 0, max(dj.max(), gr.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='greedy = dijkstra (optimal)')
    ax.set_xlabel(f'Dijkstra {title.lower()}')
    ax.set_ylabel(f'Greedy {title.lower()}')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()